# 서울시 시영주차장 30분 뒤 혼잡도 예측 (SVM)

## 1. 라이브러리와 설정값

In [57]:
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

In [58]:
THRESHOLDS = [0.50, 0.85]             # 여유 < 50% <= 보통 < 85% <= 혼잡
LEVELS = ['여유', '보통', '혼잡']
HORIZON = 30                          # 예측 간격 (분)
TOLERANCE = 6                         # 관측을 찾을 때 허용 오차 (분)
STALE_LIMIT = 20                      # 갱신이 멈춘 것으로 보는 기준 (분)
MIN_UNIQUE = 3                        # 주차 대수의 서로 다른 값이 이보다 적으면 고정값 주차장
MIN_OBS = 100                         # 고정값 여부를 판단할 최소 관측 수
TEST_RATIO = 0.2                      # 테스트로 쓸 마지막 시점의 비율
TUNE_SIZE = 5000                      # SVC, SVR 탐색에 쓸 표본 수
FIT_SIZE = 20000                      # SVC, SVR 학습에 쓸 표본 수
SEED = 42

## 2. 데이터 불러오기

| 폴더 | 내용 |
|---|---|
| `data/history` | 2026-05-25 ~ 2026-11-25, 공개 저장소 커밋 기록에서 복원한 값 |
| `data/manual` | 직접 수집한 값 |
| `data/collected` | GitHub Actions 수집 값 (있을 때만 사용) |

- `raw/`: 주차장별 현재 주차 대수
- `static/`: 주차장 정보 (주소, 요금, 운영 시각)
- 원천: 서울 열린데이터광장 OA-21709 (공공누리 1유형)
- 복원 출처: GitHub `Seoul-Park/Seoul-Park`, `Soom260/parking_information`

In [59]:
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SOURCES = ['history', 'manual', 'collected']

raw_files = [f for s in SOURCES for f in sorted((ROOT / 'data' / s / 'raw').glob('*.csv'))]
static_files = sorted([f for s in SOURCES for f in (ROOT / 'data' / s / 'static').glob('*.csv')], key=lambda f: f.name)
print('raw:', len(raw_files), '개', raw_files[0].name, '~', raw_files[-1].name)
print('static:', static_files[-1].name)

raw: 184 개 2026-05-25.csv ~ 2026-11-26.csv
static: 2026-11-26.csv


In [60]:
# 주차장 코드와 운영 시각은 문자열로 읽기 ('0900'의 앞자리 0 유지)
TIME_COLS = ['WD_OPER_BGNG_TM', 'WD_OPER_END_TM',
             'WE_OPER_BGNG_TM', 'WE_OPER_END_TM',
             'LHLDY_OPER_BGNG_TM', 'LHLDY_OPER_END_TM']

raw = pd.concat([pd.read_csv(f, dtype={'PKLT_CD': str}) for f in raw_files], ignore_index=True)
static = pd.read_csv(static_files[-1], dtype={'PKLT_CD': str, **{c: str for c in TIME_COLS}})

In [61]:
raw

,collected_at,PKLT_CD,TPKCT,NOW_PRK_VHCL_CNT,NOW_PRK_VHCL_UPDT_TM,PRK_STTS_YN
0,2026-05-25T00:56:21+09:00,1037932,91.0,32.0,2026-05-25 00:55:22,1
1,2026-05-25T00:56:21+09:00,1051043,399.0,84.0,2026-05-25 00:55:23,1
2,2026-05-25T00:56:21+09:00,1163833,59.0,30.0,2026-05-25 00:55:31,1
3,2026-05-25T00:56:21+09:00,1167546,21.0,10.0,2026-05-25 00:55:28,1
4,2026-05-25T00:56:21+09:00,1208601,9.0,4.0,2026-05-25 00:55:26,1
...,...,...,...,...,...,...
256681,2026-11-26T01:04:54+09:00,1915194,48.0,0.0,NaN,0
256682,2026-11-26T01:04:54+09:00,2877728,885.0,104.0,2026-11-26 01:03:23,1
256683,2026-11-26T01:04:54+09:00,2983079,34.0,0.0,NaN,0
256684,2026-11-26T01:04:54+09:00,3187200,1186.0,407.0,2026-11-26 01:03:23,1


In [62]:
static.head()

,collected_at,PKLT_CD,PKLT_NM,ADDR,PKLT_TYPE,PRK_TYPE_NM,OPER_SE,OPER_SE_NM,TELNO,PRK_STTS_YN,...,ADD_PRK_HR,BUS_BSC_PRK_CRG,BUS_BSC_PRK_HR,BUS_ADD_PRK_HR,BUS_ADD_PRK_CRG,DAY_MAX_CRG,SHRN_PKLT_MNG_NM,SHRN_PKLT_MNG_URL,SHRN_PKLT_YN,SHRN_PKLT_ETC
0,2026-11-26T01:04:54+09:00,171721,세종로 공영주차장(시),종로구 세종로 80-1,NW,노외 주차장,1,시간제 주차장,02-2290-6566,1,...,5.0,0.0,0.0,0.0,0.0,0.0,*,*,N,*
1,2026-11-26T01:04:54+09:00,171730,종묘주차장 공영주차장(시),종로구 훈정동 2-0,NW,노외 주차장,1,시간제 주차장,02-2290-6166,1,...,5.0,0.0,0.0,0.0,0.0,0.0,*,*,N,*
2,2026-11-26T01:04:54+09:00,171900,훈련원공원 공영주차장(시),중구 을지로5가 40-3,NW,노외 주차장,1,시간제 주차장,02-3405-4597,1,...,5.0,0.0,0.0,0.0,0.0,0.0,*,*,N,*
3,2026-11-26T01:04:54+09:00,172051,한강진역 공영주차장(시),용산구 한남동 728-27,NW,노외 주차장,1,시간제 주차장,02-795-6406,1,...,5.0,0.0,0.0,0.0,0.0,0.0,*,*,N,*
4,2026-11-26T01:04:54+09:00,172065,용산주차빌딩 공영주차장(시),용산구 한강로2가 12-9,NW,노외 주차장,1,시간제 주차장,02-2290-6014,1,...,5.0,0.0,0.0,0.0,0.0,0.0,*,*,N,*


### 2-1. 두 파일 합치기

In [63]:
STATIC_COLS = ['PKLT_CD', 'PKLT_NM', 'ADDR', 'PKLT_TYPE', 'OPER_SE_NM', 'PAY_YN',
               'BSC_PRK_CRG', 'BSC_PRK_HR'] + TIME_COLS

data = raw.merge(static[STATIC_COLS], on='PKLT_CD', how='inner', validate='many_to_one')   # 현재 없는 주차장 제외

### 2-2. 시각 변환과 시간순 정렬

In [64]:
data['collected_at'] = pd.to_datetime(data['collected_at']).dt.tz_localize(None)   # KST
data['NOW_PRK_VHCL_UPDT_TM'] = pd.to_datetime(data['NOW_PRK_VHCL_UPDT_TM'])

data = data.sort_values(['collected_at', 'PKLT_CD']).reset_index(drop=True)

### 2-3. 확인

In [65]:
print(data.head())
print(data.info())

         collected_at  PKLT_CD  TPKCT  NOW_PRK_VHCL_CNT NOW_PRK_VHCL_UPDT_TM  \
0 2026-05-25 00:56:21  1037932   91.0              32.0  2026-05-25 00:55:22   
1 2026-05-25 00:56:21  1051043  399.0              84.0  2026-05-25 00:55:23   
2 2026-05-25 00:56:21  1163833   59.0              30.0  2026-05-25 00:55:31   
3 2026-05-25 00:56:21  1167546   21.0              10.0  2026-05-25 00:55:28   
4 2026-05-25 00:56:21  1208601    9.0               4.0  2026-05-25 00:55:26   

   PRK_STTS_YN            PKLT_NM            ADDR PKLT_TYPE OPER_SE_NM PAY_YN  \
0            1  구로디지털단지역 공영주차장(시)   구로구 구로동 810-3        NW    시간제 주차장      Y   
1            1      구파발역 공영주차장(시)    은평구 진관동 70-1        NW    시간제 주차장      Y   
2            1     봉천복개3 공영주차장(시)  관악구 신림동 1467-3        NS    시간제 주차장      Y   
3            1   독산동 금천교 공영주차장(시)  금천구 독산동 1081-4        NS    시간제 주차장      Y   
4            1   청계2(북2) 공영주차장(시)    종로구 관수동 91-4        NS    시간제 주차장      Y   

   BSC_PRK_CRG  BSC_PRK_HR WD_OP

In [66]:
print('수집 횟수:', data['collected_at'].nunique())
print('기간:', data['collected_at'].min(), '~', data['collected_at'].max())
print('주차장 수:', data['PKLT_CD'].nunique())
print('결측치')
print(data.isna().sum()[lambda s: s > 0])

수집 횟수: 2137
기간: 2026-05-25 00:56:21 ~ 2026-11-26 01:04:54
주차장 수: 122
결측치
NOW_PRK_VHCL_UPDT_TM    24816
dtype: int64


## 3. 정제

| 제외 대상 | 조건 |
|---|---|
| 중복 행 | 수집 시각과 주차장 코드가 같은 행 |
| 미연계 주차장 | `PRK_STTS_YN`이 1이 아닌 행 |
| 버스전용 주차장 | `OPER_SE_NM`이 버스전용 주차장 |
| 갱신이 멈춘 행 | 갱신 시각이 20분 넘게 지난 행 |
| 소형 주차장 | 총 주차면 5면 미만 |
| 고정값 주차장 | 관측 100회 이상인데 주차 대수 값이 2가지 이하 |

In [67]:
n_before = len(data)
age = data['collected_at'] - data['NOW_PRK_VHCL_UPDT_TM']

data = data[(data['PRK_STTS_YN'] == 1)
            & (data['OPER_SE_NM'] != '버스전용 주차장')
            & (age <= pd.Timedelta(minutes=STALE_LIMIT))
            & (data['TPKCT'] >= 5)]
data = data.drop_duplicates(['collected_at', 'PKLT_CD']).reset_index(drop=True)

stats = data.groupby('PKLT_CD')['NOW_PRK_VHCL_CNT'].agg(['size', 'nunique'])
frozen = stats.index[(stats['size'] >= MIN_OBS) & (stats['nunique'] < MIN_UNIQUE)]
data = data[~data['PKLT_CD'].isin(frozen)].reset_index(drop=True)

print('정제 전:', n_before, '행')
print('고정값 주차장:', len(frozen), '곳 제외')
print('정제 후:', len(data), '행, 주차장', data['PKLT_CD'].nunique(), '곳')

정제 전: 256323 행
고정값 주차장: 31 곳 제외
정제 후: 143452 행, 주차장 68 곳


## 4. 정답 만들기

- 점유율 = 현재 주차 대수 ÷ 총 주차면 (최대 1)
- 정답 = 같은 주차장의 30분 뒤 혼잡도 (오차 6분 이내 관측)

In [68]:
def to_level(occ):
    # 점유율 -> 0(여유), 1(보통), 2(혼잡)
    return np.digitize(occ, THRESHOLDS)


def occ_at(df, minutes):
    # 같은 주차장의 minutes분 뒤(음수는 전) 점유율
    left = pd.DataFrame({'PKLT_CD': df['PKLT_CD'],
                         'time': df['collected_at'] + pd.Timedelta(minutes=minutes),
                         'row': np.arange(len(df))}).sort_values('time')
    right = df[['PKLT_CD', 'collected_at', 'occ']].rename(columns={'collected_at': 'time', 'occ': 'value'})
    merged = pd.merge_asof(left, right.sort_values('time'), on='time', by='PKLT_CD',
                           direction='nearest', tolerance=pd.Timedelta(minutes=TOLERANCE))
    return merged.sort_values('row')['value'].to_numpy()

In [69]:
data['occ'] = (data['NOW_PRK_VHCL_CNT'] / data['TPKCT']).clip(0, 1)
data['occ_future'] = occ_at(data, HORIZON)

print('현재 혼잡도 비율')
print(pd.Series(to_level(data['occ'])).map(dict(enumerate(LEVELS))).value_counts(normalize=True).round(3))
print('30분 뒤 관측이 있는 행:', data['occ_future'].notna().sum(), '/', len(data))

현재 혼잡도 비율
여유    0.561
보통    0.326
혼잡    0.114
Name: proportion, dtype: float64
30분 뒤 관측이 있는 행: 19902 / 143452


## 5. 특성 만들기

| 특성 | 내용 |
|---|---|
| `occ` | 현재 점유율 |
| `diff10`, `diff30` | 10분 전, 30분 전 대비 점유율 변화 (관측이 없으면 0, 결측 표시 추가) |
| `hour_sin`, `hour_cos` | 시각 |
| `dow` | 요일 |
| `is_offday` | 주말 또는 공휴일 |
| `is_open` | 운영 시간 여부 (`0000~2400`은 24시간, `0000~0000`은 미운영으로 가정) |
| `log_capacity` | 총 주차면 (로그) |
| `fee_per_hour` | 시간당 기본 요금 |
| `PKLT_TYPE` | 노상(NS), 노외(NW) |

In [70]:
data['diff10'] = data['occ'] - occ_at(data, -10)
data['diff30'] = data['occ'] - occ_at(data, -30)

time = data['collected_at']
minute = time.dt.hour * 60 + time.dt.minute
data['hour_sin'] = np.sin(2 * np.pi * minute / 1440)
data['hour_cos'] = np.cos(2 * np.pi * minute / 1440)
data['dow'] = time.dt.day_of_week

kr_holidays = holidays.KR(years=sorted(time.dt.year.unique().tolist()))
is_weekend = data['dow'] >= 5
is_holiday = time.dt.date.map(lambda d: d in kr_holidays).astype(bool)
data['is_offday'] = (is_weekend | is_holiday).astype(int)

data['log_capacity'] = np.log1p(data['TPKCT'])
data['fee_per_hour'] = (data['BSC_PRK_CRG'] / data['BSC_PRK_HR'].where(data['BSC_PRK_HR'] > 0) * 60).fillna(0)

In [71]:
def to_minute(s):
    # '0930' -> 570
    return s.str[:2].astype(int) * 60 + s.str[2:].astype(int)


def pick(suffix):
    # 평일(WD), 주말(WE), 공휴일(LHLDY) 중 해당하는 운영 시각
    return np.select([is_holiday, is_weekend],
                     [to_minute(data[f'LHLDY_OPER_{suffix}_TM']), to_minute(data[f'WE_OPER_{suffix}_TM'])],
                     to_minute(data[f'WD_OPER_{suffix}_TM']))


start, end = pick('BGNG'), pick('END')
same_day = (start <= minute) & (minute < end)
overnight = (start > end) & ((minute >= start) | (minute < end))   # 예: 0900~0100
data['is_open'] = (same_day | overnight).astype(int)

In [72]:
NUM_COLS = ['occ', 'diff10', 'diff30', 'hour_sin', 'hour_cos', 'is_offday',
            'is_open', 'log_capacity', 'fee_per_hour']
CAT_COLS = ['dow', 'PKLT_TYPE']
FEATURES = NUM_COLS + CAT_COLS

data[FEATURES]

,occ,diff10,diff30,hour_sin,hour_cos,is_offday,is_open,log_capacity,fee_per_hour,dow,PKLT_TYPE
0,0.351648,NaN,NaN,0.241922,0.970296,1,1,4.521789,3840.0,0,NW
1,0.210526,NaN,NaN,0.241922,0.970296,1,1,5.991465,2880.0,0,NW
2,0.508475,NaN,NaN,0.241922,0.970296,1,0,4.094345,2640.0,0,NS
3,0.611111,NaN,NaN,0.241922,0.970296,1,0,2.944439,1440.0,0,NS
4,0.493671,NaN,NaN,0.241922,0.970296,1,0,4.382027,4320.0,0,NW
...,...,...,...,...,...,...,...,...,...,...,...
143447,0.366177,NaN,NaN,0.275637,0.961262,0,1,7.266827,4320.0,3,NW
143448,0.788732,NaN,NaN,0.275637,0.961262,0,1,4.276666,1560.0,3,NW
143449,0.117514,NaN,NaN,0.275637,0.961262,0,1,6.786717,4800.0,3,NW
143450,0.343170,NaN,NaN,0.275637,0.961262,0,1,7.079184,6600.0,3,NW


## 6. 학습·테스트 나누기

- 30분 단위로 한 행씩 사용
- 시간순으로 앞 80% 시점은 학습, 뒤 20% 시점은 테스트
- 정답 시각이 테스트 기간과 겹치는 학습 행 제외

In [73]:
LAG_COLS = ['diff10', 'diff30']
table = data.dropna(subset=[c for c in FEATURES if c not in LAG_COLS] + ['occ_future']).copy()
table['slot'] = table['collected_at'].dt.floor('30min')
table = table.drop_duplicates(['slot', 'PKLT_CD']).sort_values(['collected_at', 'PKLT_CD']).reset_index(drop=True)
table['level'] = to_level(table['occ_future'])

n_slots = table['slot'].nunique()
print('사용할 행:', len(table), ' 30분 단위 시점:', n_slots)
print('변화량 관측 비율:', table[LAG_COLS].notna().mean().round(3).to_dict())
if n_slots < 48:
    raise ValueError('데이터 부족: 최소 하루(30분 단위 시점 48개) 이상 수집 후 다시 실행')

사용할 행: 19227  30분 단위 시점: 286
변화량 관측 비율: {'diff10': 0.042, 'diff30': 0.381}


In [74]:
GAP = pd.Timedelta(minutes=HORIZON + TOLERANCE)

slots = np.sort(table['slot'].unique())
first_test_slot = slots[int(len(slots) * (1 - TEST_RATIO))]
cutoff = table.loc[table['slot'] == first_test_slot, 'collected_at'].min()

train = table[table['collected_at'] + GAP < cutoff].reset_index(drop=True)
test = table[table['collected_at'] >= cutoff].reset_index(drop=True)

train_x, train_y = train[FEATURES], train['level']
test_x, test_y = test[FEATURES], test['level']

print('학습:', train_x.shape, train['collected_at'].min(), '~', train['collected_at'].max(), ' 시점:', train['slot'].nunique())
print('테스트:', test_x.shape, test['collected_at'].min(), '~', test['collected_at'].max(), ' 시점:', test['slot'].nunique())
print(pd.DataFrame({'학습': train_y.value_counts(normalize=True), '테스트': test_y.value_counts(normalize=True)})
      .reindex([0, 1, 2]).fillna(0).set_axis(LEVELS).round(3))
if train_y.nunique() < 3:
    raise ValueError('데이터 부족: 학습 데이터에 세 혼잡도가 모두 필요')

학습: (15222, 11) 2026-08-08 05:48:26 ~ 2026-10-08 02:13:53  시점: 227
테스트: (3937, 11) 2026-10-08 03:24:58 ~ 2026-11-20 22:28:41  시점: 58
       학습    테스트
여유  0.599  0.528
보통  0.309  0.340
혼잡  0.092  0.132


In [75]:
train_x

,occ,diff10,diff30,hour_sin,hour_cos,is_offday,is_open,log_capacity,fee_per_hour,dow,PKLT_TYPE
0,0.142857,NaN,NaN,0.998630,0.052336,1,1,4.521789,3840.0,5,NW
1,0.278195,NaN,NaN,0.998630,0.052336,1,1,5.991465,2880.0,5,NW
2,0.542373,NaN,NaN,0.998630,0.052336,1,0,4.094345,2640.0,5,NS
3,0.388889,NaN,NaN,0.998630,0.052336,1,0,2.944439,1440.0,5,NS
4,0.708861,NaN,NaN,0.998630,0.052336,1,0,4.382027,4320.0,5,NW
...,...,...,...,...,...,...,...,...,...,...,...
15217,0.398323,NaN,-0.003494,0.548293,0.836286,0,1,7.266827,4320.0,3,NW
15218,0.732394,NaN,0.000000,0.548293,0.836286,0,1,4.276666,1560.0,3,NW
15219,0.109605,NaN,0.000000,0.548293,0.836286,0,1,6.786717,4800.0,3,NW
15220,0.656830,NaN,-0.001686,0.548293,0.836286,0,1,7.079184,6600.0,3,NW


## 7. 인코딩과 스케일링

- 변화량 결측: `SimpleImputer` (0으로 채우고 결측 표시 추가)
- 숫자 특성: `StandardScaler`
- 범주 특성: `OneHotEncoder`
- `Pipeline`으로 묶어 학습 데이터에만 `fit`

In [76]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='constant', fill_value=0, add_indicator=True),
                          StandardScaler()), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_COLS),
])

print('변환 후 크기:', preprocess.fit_transform(train_x).shape)

변환 후 크기: (15222, 20)


## 8. 모델 학습

| 모델 | 탐색할 값 |
|---|---|
| 기준선: 최빈 클래스 | 없음 |
| 기준선: 지금 그대로 | 없음 |
| LinearSVC | `C` |
| SVC (RBF) | `C`, `gamma` |
| SVR 예측 후 구간 나누기 | `C`, `gamma`, `epsilon` |

탐색 기준: 시간순 교차검증의 macro F1

In [77]:
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit


def time_cv(df, n_splits=3):
    # 같은 시점의 주차장들이 나뉘지 않고, 정답 시각도 겹치지 않는 시간순 교차검증
    slots = np.sort(df['slot'].unique())
    folds = []
    for tr, va in TimeSeriesSplit(n_splits=n_splits).split(slots):
        valid = df['slot'].isin(slots[va])
        valid_start = df.loc[valid, 'collected_at'].min()
        fit = df['slot'].isin(slots[tr]) & (df['collected_at'] + GAP < valid_start)
        folds.append((np.flatnonzero(fit), np.flatnonzero(valid)))
    return folds


def take_sample(df, n):
    # n개를 뽑아 시간순으로 정렬
    if len(df) > n:
        df = df.sample(n=n, random_state=SEED)
    return df.sort_values(['collected_at', 'PKLT_CD']).reset_index(drop=True)


tune = take_sample(train, TUNE_SIZE)
fit = take_sample(train, FIT_SIZE)
predictions, cv_scores = {}, {}

### 8-1. 기준선

In [78]:
predictions['최빈 클래스'] = np.full(len(test), train_y.mode()[0])
predictions['지금 그대로'] = to_level(test['occ'])

### 8-2. LinearSVC

In [79]:
from sklearn.svm import SVC, SVR, LinearSVC

linear = GridSearchCV(
    make_pipeline(preprocess, LinearSVC(class_weight='balanced', max_iter=10000, random_state=SEED)),
    {'linearsvc__C': [0.01, 0.1, 1, 10, 100]},
    cv=time_cv(train), scoring='f1_macro', n_jobs=-1)
linear.fit(train_x, train_y)

predictions['LinearSVC'] = linear.predict(test_x)
cv_scores['LinearSVC'] = linear.best_score_
print(linear.best_params_, round(linear.best_score_, 4))

{'linearsvc__C': 0.1} 0.8685


### 8-3. SVC (RBF 커널)

In [80]:
svc_search = GridSearchCV(
    make_pipeline(preprocess, SVC(kernel='rbf', class_weight='balanced')),
    {'svc__C': [0.1, 1, 10, 100], 'svc__gamma': [0.01, 0.03, 0.1, 0.3]},
    cv=time_cv(tune), scoring='f1_macro', n_jobs=-1)
svc_search.fit(tune[FEATURES], tune['level'])

svc = svc_search.best_estimator_.fit(fit[FEATURES], fit['level'])
predictions['SVC'] = svc.predict(test_x)
cv_scores['SVC'] = svc_search.best_score_
print(svc_search.best_params_, round(svc_search.best_score_, 4))

{'svc__C': 10, 'svc__gamma': 0.01} 0.8955


### 8-4. SVR 예측 후 구간 나누기

In [81]:
def binned_f1(model, x, occ_future):
    # 예측 점유율을 혼잡도로 나눈 뒤 macro F1
    return f1_score(to_level(occ_future), to_level(model.predict(x)), average='macro', zero_division=0)


svr_search = GridSearchCV(
    make_pipeline(preprocess, SVR(kernel='rbf')),
    {'svr__C': [0.1, 1, 10], 'svr__gamma': [0.01, 0.1], 'svr__epsilon': [0.02, 0.05]},
    cv=time_cv(tune), scoring=binned_f1, n_jobs=-1)
svr_search.fit(tune[FEATURES], tune['occ_future'])

svr = svr_search.best_estimator_.fit(fit[FEATURES], fit['occ_future'])
svr_occ = np.clip(svr.predict(test_x), 0, 1)
predictions['SVR'] = to_level(svr_occ)
cv_scores['SVR'] = svr_search.best_score_
print(svr_search.best_params_, round(svr_search.best_score_, 4))
print('점유율 평균절대오차  SVR:', round(np.abs(svr_occ - test['occ_future']).mean(), 4),
      ' 지금 그대로:', round(np.abs(test['occ'] - test['occ_future']).mean(), 4))

{'svr__C': 0.1, 'svr__epsilon': 0.02, 'svr__gamma': 0.01} 0.9213
점유율 평균절대오차  SVR: 0.0253  지금 그대로: 0.024


## 9. 예측과 평가

- macro F1, 정확도, 클래스별 재현율
- 큰 오차: 여유와 혼잡을 서로 바꿔 예측한 비율
- 변화 구간: 30분 사이 혼잡도가 바뀐 행

In [82]:
from sklearn.metrics import accuracy_score, classification_report, recall_score


def evaluate(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    if len(y) == 0:
        return {}
    recall = recall_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {'macro F1': f1_score(y, pred, labels=np.unique(y), average='macro', zero_division=0),
            '정확도': accuracy_score(y, pred),
            **{f'재현율 {name}': recall[i] for i, name in enumerate(LEVELS)},
            '큰 오차': (np.abs(y - pred) == 2).mean()}


result = pd.DataFrame({name: evaluate(test_y, pred) for name, pred in predictions.items()}).T
result.insert(0, 'CV macro F1', pd.Series(cv_scores))
result.round(3)

,CV macro F1,macro F1,정확도,재현율 여유,재현율 보통,재현율 혼잡,큰 오차
최빈 클래스,NaN,0.230,0.528,1.000,0.000,0.000,0.132
지금 그대로,NaN,0.934,0.942,0.962,0.912,0.936,0.001
LinearSVC,0.868,0.907,0.917,0.951,0.882,0.867,0.001
SVC,0.895,0.909,0.919,0.926,0.892,0.963,0.001
SVR,0.921,0.929,0.939,0.974,0.900,0.902,0.001


### 9-1. 변화 구간

In [83]:
changed = to_level(test['occ']) != test_y.to_numpy()
print('변화 구간:', changed.sum(), '행', f'({changed.mean():.1%})')

pd.DataFrame({name: evaluate(test_y[changed], pred[changed]) for name, pred in predictions.items()}).T.round(3)

변화 구간: 229 행 (5.8%)


,macro F1,정확도,재현율 여유,재현율 보통,재현율 혼잡,큰 오차
최빈 클래스,0.169,0.341,1.000,0.000,0.000,0.144
지금 그대로,0.000,0.000,0.000,0.000,0.000,0.013
LinearSVC,0.228,0.271,0.385,0.254,0.061,0.009
SVC,0.266,0.271,0.141,0.305,0.455,0.013
SVR,0.212,0.240,0.359,0.203,0.091,0.013


### 9-2. 모델별 상세 결과

In [84]:
for name in ['LinearSVC', 'SVC', 'SVR']:
    print('=====', name)
    print(classification_report(test_y, predictions[name], labels=[0, 1, 2], target_names=LEVELS,
                                digits=3, zero_division=0))
    print(pd.crosstab(pd.Categorical.from_codes(test_y, LEVELS), pd.Categorical.from_codes(predictions[name], LEVELS),
                      rownames=['실제'], colnames=['예측'], dropna=False))
    print()

===== LinearSVC
              precision    recall  f1-score   support

          여유      0.940     0.951     0.946      2080
          보통      0.875     0.882     0.879      1338
          혼잡      0.932     0.867     0.898       519

    accuracy                          0.917      3937
   macro avg      0.916     0.900     0.907      3937
weighted avg      0.917     0.917     0.917      3937

예측    여유    보통   혼잡
실제                 
여유  1979    99    2
보통   127  1180   31
혼잡     0    69  450

===== SVC
              precision    recall  f1-score   support

          여유      0.978     0.926     0.951      2080
          보통      0.875     0.892     0.884      1338
          혼잡      0.829     0.963     0.891       519

    accuracy                          0.919      3937
   macro avg      0.894     0.927     0.909      3937
weighted avg      0.923     0.919     0.920      3937

예측    여유    보통   혼잡
실제                 
여유  1926   152    2
보통    43  1194  101
혼잡     1    18  500

===== SVR


## 10. 결과 저장

In [85]:
submission = test[['collected_at', 'PKLT_CD', 'PKLT_NM', 'occ', 'occ_future']].copy()
submission['실제'] = [LEVELS[i] for i in test_y]
for name, pred in predictions.items():
    submission[name] = [LEVELS[i] for i in pred]

out_dir = ROOT / 'data' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_dir / 'leejy_predictions.csv', index=False, encoding='utf-8-sig')

In [86]:
submission

,collected_at,PKLT_CD,PKLT_NM,occ,occ_future,실제,최빈 클래스,지금 그대로,LinearSVC,SVC,SVR
0,2026-10-08 03:24:58,1037932,구로디지털단지역 공영주차장(시),0.241758,0.219780,여유,여유,여유,여유,여유,여유
1,2026-10-08 03:24:58,1051043,구파발역 공영주차장(시),0.122807,0.122807,여유,여유,여유,여유,여유,여유
2,2026-10-08 03:24:58,1163833,봉천복개3 공영주차장(시),0.576271,0.576271,보통,여유,보통,보통,보통,보통
3,2026-10-08 03:24:58,1234305,가마산고가밑 공영주차장(시),0.500000,0.500000,보통,여유,보통,보통,보통,보통
4,2026-10-08 03:24:58,1247636,신설동 공영주차장(시),0.544304,0.531646,보통,여유,보통,보통,보통,보통
...,...,...,...,...,...,...,...,...,...,...,...
3932,2026-11-20 22:28:41,173867,천호역 공영주차장(시),0.450734,0.424878,여유,여유,여유,여유,여유,여유
3933,2026-11-20 22:28:41,1913393,장안1동 공영주차장(시),0.802817,0.760563,보통,여유,보통,보통,보통,보통
3934,2026-11-20 22:28:41,2877728,양재역 공영주차장(시),0.179661,0.167232,여유,여유,여유,여유,여유,여유
3935,2026-11-20 22:28:41,3187200,반포천 공영주차장(파미에)(시),0.376054,0.363406,여유,여유,여유,여유,여유,여유
